# 10 · Train Fold 0  —  FIXED VERSION

**Bugs fixed vs original:**
1. `trainer.py` — gradient clipping added (was causing `loss=inf`)
2. `trainer.py` — AMP API updated (`torch.amp` instead of `torch.cuda.amp`)
3. `trainer.py` — nan/inf guard on loss
4. `trainer.py` — LR warmup added
5. `Cell 5` — passes `max_grad_norm` and `warmup_iters` to trainer
6. `Cell 5` — `grad_accum_steps=4` to match paper's effective batch=64


In [ ]:
# Cell 1: Imports and Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import torch
import numpy as np
import pandas as pd

from src.models.hybrid_model import HybridChagasModel
from src.training.dataset import create_dataloaders
from src.training.trainer import ChagasTrainer

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.2f} GB')
    if mem_gb < 8:
        print('⚠️  GPU < 8GB — using batch_size=16 + grad_accum=4 (effective=64)')
else:
    print('⚠️  WARNING: No GPU. Training will be very slow.')

print('\n✓ Imports OK')

In [ ]:
# Cell 2: Configuration
FOLD = 0  # Change to 1,2,3,4 for other folds

DATA_DIR     = project_root / 'data' / 'processed'
METADATA_CSV = DATA_DIR / 'metadata' / 'combined_5fold.csv'
IMAGES_DIR   = DATA_DIR / '2d_images'
SIGNALS_DIR  = DATA_DIR / '1d_signals_100hz'

if not METADATA_CSV.exists():
    raise FileNotFoundError(f'Metadata CSV not found: {METADATA_CSV}')
print(f'✓ Metadata CSV: {METADATA_CSV}')

CHECKPOINT_DIR = project_root / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MAE_CHECKPOINT   = CHECKPOINT_DIR / 'mae_2d_pretrained.pt'
STMEM_CHECKPOINT = CHECKPOINT_DIR / 'stmem_1d_pretrained.pt'

# ── Training config ───────────────────────────────────────────────────
BATCH_SIZE   = 16          # fits 6GB GPU
GRAD_ACCUM   = 4           # effective batch = 16 × 4 = 64  (matches paper)
NUM_WORKERS  = 4
USE_AMP      = True

# Phase 1 — FM frozen
PHASE1_ITERATIONS = 2000
PHASE1_LR         = 2e-4

# Phase 2 — FM unfrozen
PHASE2_ITERATIONS = 12000
PHASE2_LR_HIGH    = 2e-4   # classifier + REPA
PHASE2_LR_LOW     = 2e-5   # FM + 2D-ViT (10× lower)

# FIX params
MAX_GRAD_NORM = 1.0     # gradient clipping threshold
WARMUP_ITERS  = 200     # linear warmup steps

RESUME_FROM = None  # set to path string to resume

print(f'\n✓ Config — Fold {FOLD}:')
print(f'  Batch size:       {BATCH_SIZE}')
print(f'  Grad accum:       {GRAD_ACCUM}  (effective batch={BATCH_SIZE*GRAD_ACCUM})')
print(f'  Max grad norm:    {MAX_GRAD_NORM}  ← prevents loss=inf')
print(f'  Warmup iters:     {WARMUP_ITERS}')
print(f'  Phase 1:          {PHASE1_ITERATIONS} iters')
print(f'  Phase 2:          {PHASE2_ITERATIONS} iters')
print(f'  Total:            {PHASE1_ITERATIONS+PHASE2_ITERATIONS} iters')

In [ ]:
# Cell 3: Create Dataloaders
print('Creating dataloaders...')

train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,
    augment_train=True
)

print('\nTesting first batch...')
batch = next(iter(train_loader))
print(f'  image:  {batch["image"].shape}')   # (16, 3, 24, 2048)
print(f'  signal: {batch["signal"].shape}')  # (16, 12, 1000)
print(f'  age:    {batch["age"].shape}')     # (16,)
print(f'  sex:    {batch["sex"].shape}')     # (16,)
print(f'  label:  {batch["label"].shape}')  # (16,)

# Sanity check — labels should be in [0,1]
assert batch['label'].min() >= 0 and batch['label'].max() <= 1, 'Labels out of [0,1]!'
# Sanity check — no NaN in signals
assert not torch.isnan(batch['signal']).any(), 'NaN detected in signals!'
print('\n✓ Dataloaders ready — no NaN, labels in [0,1]')

In [ ]:
# Cell 4: Create Model
print('Creating model...')

model = HybridChagasModel(
    img_size=(24, 2048),
    patch_size_2d=(8, 64),
    num_leads=12,
    seq_len_1d=1000,
    patch_size_1d=50,
    embed_dim=768,
    depth=12,
    num_heads=12,
    use_aol=True,
    use_demographics=True
)

if MAE_CHECKPOINT.exists():
    print(f'✓ Loading MAE weights from {MAE_CHECKPOINT}')
    model.vit_2d.load_mae_pretrained(str(MAE_CHECKPOINT))
else:
    print('⚠️  No MAE checkpoint — 2D-ViT trains from scratch (lower expected score)')

if STMEM_CHECKPOINT.exists():
    print(f'✓ Loading ST-MEM weights from {STMEM_CHECKPOINT}')
    model.vit_1d_fm.load_stmem_pretrained(str(STMEM_CHECKPOINT))
else:
    print('⚠️  No ST-MEM checkpoint — 1D-ViT FM trains from scratch (lower expected score)')

model = model.to(device)

# Quick forward pass sanity check
with torch.no_grad():
    test_out = model(
        batch['image'].to(device),
        batch['signal'].to(device),
        batch['age'].to(device),
        batch['sex'].to(device),
    )
    assert torch.isfinite(test_out['logits']).all(), 'Model produces non-finite logits at init!'
    print(f'\n✓ Forward pass OK — logits shape: {test_out["logits"].shape}')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✓ Model ready:')
print(f'  Total params:     {total_params:,}')
print(f'  Trainable params: {trainable_params:,}')

In [ ]:
# Cell 5: Create Trainer  ── v6 params
print('Creating trainer...')

trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,    # 2000
    phase2_iterations=PHASE2_ITERATIONS,    # 12000
    phase1_lr=PHASE1_LR,                    # 2e-4
    phase2_lr_high=PHASE2_LR_HIGH,          # 2e-4
    phase2_lr_low=PHASE2_LR_LOW,            # 2e-5
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP,
    val_every_n_iters=500,
    max_grad_norm=MAX_GRAD_NORM,            # 1.0  — prevents loss=inf
    warmup_iters=WARMUP_ITERS,             # 200  — stabilises early training
    # ── v6 PARAMS (key changes) ──────────────────────────────────────
    phase1_grad_accum=4,    # Phase 1: FM frozen, cheap → accum=4 matches paper eff.batch=64
    phase2_grad_accum=1,    # Phase 2: ALL 173M params → accum=1 → ~8h not ~30h
    val_subset_size=3000,   # Stratified subset (FIX: no more "only one class")
    val_n_permutations=1000,# Fast mid-training metric (10000 used at phase end)
)

print('✓ Trainer ready')
print(f'  Val subset: {trainer.val_subset_size} stratified samples (both classes guaranteed)')
print(f'  Phase 1 ETA: ~35 min  |  Phase 2 ETA: ~8-10 hours')


In [ ]:
# Cell 6: TRAIN
print('\nStarting training...\n')

metrics = trainer.train(fold=FOLD, resume_from=RESUME_FROM)

print(f'\n' + '='*70)
print(f' Final Results — Fold {FOLD}')
print('='*70)
print(f'  TPR@5%:  {metrics["tpr_5pct"]:.4f}  ← PRIMARY METRIC')
print(f'  AUROC:   {metrics["auroc"]:.4f}')
print(f'  AUPRC:   {metrics.get("auprc",0):.4f}')
print('='*70)

if metrics['tpr_5pct'] >= 0.42:
    print('✅ TARGET ACHIEVED (≥0.42)')
elif metrics['tpr_5pct'] >= 0.35:
    print('⚠️  Good progress — below target 0.42')
elif metrics['tpr_5pct'] >= 0.20:
    print('⚠️  Training worked (no divergence) but score low — try pretraining')
else:
    print('❌ Low score — check data loading and normalisation')

gap = 0.445 - metrics['tpr_5pct']
if gap <= 0:
    print('🎉 Matches/beats top team (0.445)!')
else:
    print(f'  Gap to top team (0.445): {gap:.4f}')

In [ ]:
# Cell 7: Save Results + Plot Training Curve
import matplotlib.pyplot as plt

results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_csv = CHECKPOINT_DIR / f'fold{FOLD}_results.csv'
results_df.to_csv(results_csv, index=False)
print(f'✓ Results saved: {results_csv}')

# Plot training curve
history = trainer.history
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

if history['train_loss']:
    axes[0].plot(history['train_loss'])
    axes[0].set_title('Train Loss (50-iter smooth)')
    axes[0].set_xlabel('Iteration')
    axes[0].axvline(x=2000, color='r', linestyle='--', label='Phase 2 start')
    axes[0].legend()

if history['val_tpr_5pct']:
    val_iters = [500 * (i+1) for i in range(len(history['val_tpr_5pct']))]
    axes[1].plot(val_iters, history['val_tpr_5pct'], 'g-o')
    axes[1].set_title('Val TPR@5%')
    axes[1].set_xlabel('Iteration')
    axes[1].axhline(y=0.42, color='r', linestyle='--', label='Target 0.42')
    axes[1].legend()

if history['grad_norm']:
    axes[2].plot(history['grad_norm'][:2000], alpha=0.5)
    axes[2].set_title('Gradient Norm (Phase 1)')
    axes[2].set_xlabel('Iteration')
    axes[2].axhline(y=1.0, color='r', linestyle='--', label='Clip threshold')
    axes[2].legend()

plt.tight_layout()
plot_path = CHECKPOINT_DIR / f'fold{FOLD}_training_curve.png'
plt.savefig(plot_path, dpi=100)
plt.show()
print(f'✓ Training curve saved: {plot_path}')

print('\n' + '='*62)
print('NEXT STEPS:')
print('  1. Run folds 1-4 (change FOLD = 1,2,3,4)')
print('  2. Run evaluation_complete.ipynb for ensemble metrics')
print('  3. If score < 0.35, run MAE pretraining first')
print('='*62)